In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from pathlib import Path

REPORT_FIGURES_DIR = Path("../docs/report_figures")
REPORT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_plotly_figure(fig, name):
    path = REPORT_FIGURES_DIR / name
    fig.write_image(path)
    return path


In [4]:
import sys
# !{sys.executable} -m pip install numpy pandas plotly nbformat kaleido


In [5]:
csv_path = Path("/ros2_ws/galo_odometry_error_rep.csv")
if not csv_path.exists():
    csv_path = Path("galo_odometry_error_rep.csv")
data = pd.read_csv(csv_path)
data = data.sort_values(by='odom_time')
data.head()


,odom_time,gt_time,gt_x,gt_y,gt_z,gt_roll,gt_pitch,gt_yaw,odom_x,odom_y,...,final_y,planar_dx,planar_dy,final_dx,final_dy,planar_valid,planar_gate_ok,planar_matches,planar_mean_residual,gnss_corrected
0,1.755181e+09,1.755181e+09,74.924139,765.330794,217.677640,-0.011974,0.018155,-1.194871,79.685500,753.869383,...,753.869383,0.0,0.0,-0.000156,-0.000079,0,0,0,0.0,0
1,1.755181e+09,1.755181e+09,78.717302,756.117017,217.932165,0.008501,0.009196,-1.171111,80.688553,751.626815,...,751.626815,0.0,0.0,-0.000122,-0.000294,0,0,0,0.0,1
2,1.755181e+09,1.755181e+09,78.820185,755.855855,217.940688,0.009599,0.010231,-1.167849,80.972037,750.904355,...,750.904355,0.0,0.0,-0.000148,-0.000260,0,0,0,0.0,0
3,1.755181e+09,1.755181e+09,79.030417,755.369633,217.958427,0.011970,0.011879,-1.165645,81.259327,750.180588,...,750.180588,0.0,0.0,-0.000193,-0.000225,0,0,0,0.0,0
4,1.755181e+09,1.755181e+09,79.312714,754.736154,217.970704,0.014522,0.012374,-1.162950,81.545146,749.468699,...,749.468699,0.0,0.0,-0.000231,-0.000218,0,0,0,0.0,0


In [6]:
data.columns

Index(['odom_time', 'gt_time', 'gt_x', 'gt_y', 'gt_z', 'gt_roll', 'gt_pitch',
       'gt_yaw', 'odom_x', 'odom_y', 'odom_z', 'odom_roll', 'odom_pitch',
       'odom_yaw', 'error_x', 'error_y', 'error_z', 'error_roll',
       'error_pitch', 'error_yaw', 'min_error_x', 'min_error_y', 'min_error_z',
       'min_error_roll', 'min_error_pitch', 'min_error_yaw', 'max_error_x',
       'max_error_y', 'max_error_z', 'max_error_roll', 'max_error_pitch',
       'max_error_yaw', 'pred_yaw', 'planar_yaw', 'final_yaw',
       'planar_yaw_delta', 'pred_x', 'pred_y', 'planar_x', 'planar_y',
       'final_x', 'final_y', 'planar_dx', 'planar_dy', 'final_dx', 'final_dy',
       'planar_valid', 'planar_gate_ok', 'planar_matches',
       'planar_mean_residual', 'gnss_corrected'],
      dtype='object')

In [7]:
features = [
  'gt', 'odom'
]
fig = go.Figure()

for f in features:
  fig.add_trace(
    go.Scatter(
      x = data[f'{f}_x'],
      y = data[f'{f}_y'],
      name=f
    )
  )
fig.update_yaxes(scaleanchor="x", scaleratio=1, title='y, m')
fig.update_xaxes(title='x, m')
fig.update_layout(title = "map")
save_plotly_figure(fig, "trajectory_map.png")
fig.show()

In [8]:
sources = [
  'error'
]
features = [
  'x', 'y', 'z', 'yaw', 'pitch', 'roll'
]
y_axis_titles = [
  f'{f}, m' if i < 3 else f'{f}, rad' for i, f in enumerate(features)
]
fig = make_subplots(
  cols=1, rows=len(features),
  subplot_titles=features,
  row_titles=y_axis_titles,
  shared_xaxes=True
)

for r, f in enumerate(features):
  for s in sources:
    fig.add_trace(
      go.Scatter(
        x = data['gt_time'] - data['gt_time'].min(),
        y = data[f'{s}_{f}'],
        name=f"{s}_{f}"
      ), row= r + 1, col=1
    )

max_erros = data[[f'max_error_{f}'for f in features]].loc[[data.index[-1]]].to_dict()

max_erros = {key : round(value[data.index[-1]], 3)  for i, (key, value) in enumerate(max_erros.items())} 
# print(max_erros)
fig.update_layout(title = f"Errors<br>{max_erros}", height=800, width=1200)
save_plotly_figure(fig, "odometry_errors.png")
fig.show()

In [9]:
# sources = [
#   'gt', 'odom'
# ]
# features = [
#   'x', 'y', 'z', 'yaw', 'pitch', 'roll'
# ]
# y_axis_titles = [
#   f'{f}, m' if i < 3 else f'{f}, rad' for i, f in enumerate(features)
# ]
# fig = make_subplots(
#   cols=1, rows=len(features),
#   subplot_titles=features,
#   row_titles=y_axis_titles,
#   shared_xaxes=True
# )

# for r, f in enumerate(features):
#   for s in sources:
#     fig.add_trace(
#       go.Scatter(
#         x = data['gt_time'],
#         y = data[f'{s}_{f}'],
#         name=f"{s}_{f}"
#       ), row= r + 1, col=1
#     )

# fig.update_layout(title = "Pose comparion", height=700)
# fig.show()